In [1]:
!pip install --upgrade pandas --quiet

In [2]:
import pandas as pd
import numpy as np

In [3]:
import os
import requests
import zipfile
import shutil
import subprocess

In [4]:
url = "https://www.mds.gov.br/webarquivos/publicacao/sagi/microdados/02_bolsa_familia/aibf2.zip"
zip_path = "aibf2.zip"
extract_dir = "raw"
os.makedirs(extract_dir, exist_ok=True)

In [5]:
# ==============================
# 1. baixar dados
# ==============================

print("Baixando arquivo...")
with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open(zip_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

Baixando arquivo...


In [6]:
# ==============================
# 2. extrair zip e apagá-lo
# ==============================

print("Extraindo ZIP inicial...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

os.remove(zip_path)
print("ZIP inicial removido.")

Extraindo ZIP inicial...
ZIP inicial removido.


In [7]:
# ==============================
# 3. instalar o RAR (manual + verificação)
# ==============================

def check_rar():
    try:
        subprocess.run(["unrar"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        return True
    except FileNotFoundError:
        return False

if not check_rar():
    print("\n⚠️ RAR não encontrado.")
    print("Instale manualmente:")
    print("- Windows: https://www.win-rar.com/")
    print("- Linux (Debian/Ubuntu): sudo apt install unrar")
    print("- macOS (brew): brew install unrar\n")

In [8]:
# ==============================
# 4. extrair 'Microdados para analise de impacto.rar'
# ==============================

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        if file.endswith(".rar") and "Microdados" in file:
            rar_path = os.path.join(root, file)
            print(f"Extraindo {file}...")

            subprocess.run(["unrar", "x", rar_path, root], check=False)

            os.remove(rar_path)
            print(f"{file} removido.")

Extraindo Microdados para analise descritiva_dados intermediarios.rar...

UNRAR 7.22 freeware      Copyright (c) 1993-2026 Alexander Roshal

Extracting from raw/Microdados para analise descritiva_dados intermediarios.rar

Extracting  raw/Section 16.zip                                        OK 
Extracting  raw/_Consumption Aggregates.zip                           OK 
Extracting  raw/_Misc.zip                                             OK 
Extracting  raw/01_Sintaxes.zip                                       OK 
Extracting  raw/02_Tabelas.zip                                        OK 
Extracting  raw/Section 03.zip                                        OK 
Extracting  raw/Section 04.zip                                        OK 
Extracting  raw/Section 05.zip                                        OK 
Extracting  raw/Section 06.zip                                        OK 
Extracting  raw/Section 07.zip                                        OK 
Extracting  raw/Section 08.zip        


ERROR: Bad archive raw/__MACOSX/._Microdados para analise descritiva_dados intermediarios.rar


In [9]:
# ==============================
# 5.  localizar variaveis.zip e education.zip e extrair esses zips
# ==============================

zip_targets = ["education.zip", "social capital.zip", ]

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        if file.endswith('.zip'):
            zip_file_path = os.path.join(root, file)
            if file in zip_targets:
                print(f"Extraindo {file}...")
    
                with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
                    zip_ref.extractall(root)

            os.remove(zip_file_path)
            print(f"{file} removido.")

print("\n✅ Processo concluído!")

Section 08.zip removido.
Section 09.zip removido.
Painel_Longitudinal_2005-2009.zip removido.
prenatal care.zip removido.
Dados_Brutos_2009.zip removido.
Extraindo education.zip...
education.zip removido.
anthropometry.zip removido.
descricao das variaveis.zip removido.
Extraindo social capital.zip...
social capital.zip removido.
_Consumption Aggregates.zip removido.
womens decision making.zip removido.
_Misc.zip removido.
01_Sintaxes.zip removido.
vaccinations.zip removido.
02_Tabelas.zip removido.
Dados_Brutos_2005.zip removido.
Section 15.zip removido.
birthweight.zip removido.
Section 14.zip removido.
labor supply.zip removido.
Section 16.zip removido.
Section 03.zip removido.
Section 07.zip removido.
Section 13.zip removido.
Section 12.zip removido.
Section 06.zip removido.
child labor.zip removido.
Section 10.zip removido.
Section 04.zip removido.
Section 05.zip removido.
Section 11.zip removido.

✅ Processo concluído!


In [10]:
# ==============================
# 6. manter apenas arquivos específicos
# ==============================

arquivos_permitidos_day_03 = [
    os.path.join("education", "education.dta"),
    os.path.join("education", "pscores.dta"),
    os.path.join("social capital", "1_domicilios_COMPLETE.dta")
]

print("Filtrando arquivos...")

arquivos_mantidos_day03 = []

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        caminho_completo = os.path.join(root, file)
        caminho_relativo = os.path.relpath(caminho_completo, extract_dir)

        if caminho_relativo in arquivos_permitidos_day_03:
            arquivos_mantidos_day03.append(caminho_completo)
        else:
            os.remove(caminho_completo)

Filtrando arquivos...


In [11]:
# ==============================
# 7. criar pasta "day 03" e mover arquivos
# ==============================

destino = os.path.join(extract_dir, "day 03")
os.makedirs(destino, exist_ok=True)

print("Movendo arquivos selecionados...")

for arquivo in arquivos_mantidos_day03:
    nome_arquivo = os.path.basename(arquivo)
    novo_caminho = os.path.join(destino, nome_arquivo)
    shutil.move(arquivo, novo_caminho)

Movendo arquivos selecionados...


In [12]:
# ==============================
# 8. remover diretórios vazios
# ==============================

print("Removendo diretórios vazios...")

for root, dirs, files in os.walk(extract_dir, topdown=False):
    for d in dirs:
        dir_path = os.path.join(root, d)
        if not os.listdir(dir_path):
            os.rmdir(dir_path)

print("✅ Organização concluída! Arquivos estão em 'day 03'.")

Removendo diretórios vazios...
✅ Organização concluída! Arquivos estão em 'day 03'.


In [13]:
# ======================================================================
# 1. Carregar as duas bases da AIBF II
# ======================================================================
edu = pd.read_stata("raw/day 03/education.dta", convert_categoricals=False)
dom = pd.read_stata("raw/day 03/1_domicilios_COMPLETE.dta", convert_categoricals=False)

In [14]:
os.makedirs('clean/day 03/', exist_ok=True)

In [15]:
# ======================================================================
# 2. ESCOLARIDADE DO CHEFE — construída a partir da base de pessoas.
#    s02a5 = relação com o responsável; valor 1 = é o próprio responsável.
#    educ  = nível de educação do indivíduo.
# ======================================================================
chefes = edu[edu["s02a5"] == 1][["cod_dtm", "educ"]].copy()
chefes = chefes.rename(columns={"educ": "educ_chefe"})
# se houver mais de um "responsável" por domicílio, fica a maior escolaridade
chefes = chefes.groupby("cod_dtm", as_index=False)["educ_chefe"].max()

# ======================================================================
# 3. COVARIÁVEIS DE DOMICÍLIO — tamanho e qualidade da habitação
# ======================================================================
dom_slim = dom[["cod_dtm", "s02a1", "s02a2",
                "s01x10", "s01x11", "s01x13", "s01x16", "s11a2"]].copy()

dom_slim = dom_slim.rename(columns={
    "s02a1":  "n_moradores",      # tamanho do domicílio
    "s02a2":  "n_familias",       # nº de famílias no domicílio
    "s01x10": "n_comodos",
    "s01x11": "n_dormitorios",
    "s01x13": "n_banheiros",
    "s01x16": "agua_canalizada",  # 1/2
    "s11a2":  "mae_resp_le",      # mãe do responsável sabe ler
})

# ======================================================================
# 4. Base de crianças: manter as colunas que a Aula 03 usa
# ======================================================================
cols_edu = ["cod_dtm", "s00a1", "s02af", "age", "s02ad", "s02a17", "s02a21",
            "s02a10", "s02a5", "s03a2", "s03a22", "s03a24",
            "attend_now", "dropout", "p_attend7", "if_miss30", "grade_now",
            "educ", "source"]
base = edu[cols_edu].copy()

# ======================================================================
# 5. Merge: criança + escolaridade do chefe + covariáveis do domicílio
# ======================================================================
base = base.merge(chefes,   on="cod_dtm", how="left")
base = base.merge(dom_slim, on="cod_dtm", how="left")

In [16]:
base["estrato_amostral"] = base["source"].map({1: "beneficiario", 0: "controle"})

In [17]:
base.to_csv("clean/day 03/aula03_completo.csv", index=False, encoding="utf-8")

In [18]:
# 6. Sortear domicílios ESTRATIFICANDO por 'source' (beneficiário / controle)
#    Em vez de 400 aleatórios, ~250 de cada grupo para garantir variação.
doms_por_estrato = (base.dropna(subset=["source"]).groupby("cod_dtm")["source"].first())

dom_benef   = doms_por_estrato[doms_por_estrato == 1].index
dom_control = doms_por_estrato[doms_por_estrato == 0].index

rng = np.random.RandomState(42)
sel_benef   = pd.Series(dom_benef).sample(250, random_state=42)
sel_control = pd.Series(dom_control).sample(min(250, len(dom_control)), random_state=42)
doms_sel = pd.concat([sel_benef, sel_control])

extrato = base[base["cod_dtm"].isin(doms_sel)].copy()
extrato.to_csv("clean/day 03/aula03_extrato.csv", index=False, encoding="utf-8")

In [19]:
ps = pd.read_stata("raw/day 03/pscores.dta", convert_categoricals=False)
ps = ps[["cod_dtm","tr1","pscore_tr1","tr2r2","pscore_tr2r2","weight","region"]]
ps.to_csv("clean/day 03/pscores_completo.csv", index=False, encoding="utf-8")

In [20]:
ps_extrato = ps[ps["cod_dtm"].isin(extrato["cod_dtm"].unique())]
ps_extrato.to_csv("clean/day 03/pscores_extrato.csv", index=False, encoding="utf-8")